<a href="https://colab.research.google.com/github/naeorii/estudos-analise-dados-python/blob/main/DataCleaningIntroductionAndCleaningWithDataFrames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

#**Handing Missing Data with Pandas**
pandas borrow all the capabilities from numpy selection + adds a number of convenient methods to handle missing values. Let's see one at a time:

##**Pandas utility functions**

Similarly to `numpy`, pandas also has a few utility functions to identify and detect null values:

In [2]:
# isnull() = função do Pandas que verifica se um valor está ausente
#NaN, Not a Number e é usado para representar um valor ausente
pd.isnull(np.nan)

True

In [3]:
pd.isnull(None)

True

In [4]:
pd.isna(np.nan)

True

In [5]:
pd.isna(None)

True

The opposite ones also exist:

In [6]:
pd.notnull(None)

False

In [7]:
pd.notnull(np.nan)

False

In [8]:
pd.notnull(3)

True

These functions also work with Series and `DataFrame`s:

In [9]:
pd.isnull(pd.Series([1, np.nan, 7]))

,0
0,False
1,True
2,False


In [10]:
pd.isnull(pd.DataFrame({'Coluna A': [1, np.nan, 7],
                        'Coluna B': [np.nan, 2, 3],
                        'Coluna C': [np.nan, 2, np.nan]}))

,Coluna A,Coluna B,Coluna C
0,False,True,True
1,True,False,False
2,False,False,True


##**Pandas Operation with Missing Values**
Pandas manages misisng values more gracefully than numpy. nan will no longer behave as "viruses", and operation will just ignore them completely:

In [11]:
pd.Series([1, 2, np.nan]).count()

np.int64(2)

In [12]:
pd.Series([1, 2, np.nan]).sum()

np.float64(3.0)

In [13]:
pd.Series([1, 2, np.nan]).mean()

np.float64(1.5)

##**Filtering missing data**
As we saw with `numpy`, we could combine boolean selection + `pd.isnull` to filter out those `nan`s and null values:

In [14]:
s = pd.Series([1, 2, 3, np.nan, np.nan, 4])

In [15]:
pd.notnull(s)

,0
0,True
1,True
2,True
3,False
4,False
5,True


In [16]:
pd.notnull(s).sum()

np.int64(4)

In [17]:
s[pd.notnull(s)]

,0
0,1.0
1,2.0
2,3.0
5,4.0


In [18]:
s.isnull()

,0
0,False
1,False
2,False
3,True
4,True
5,False


In [19]:
s.notnull()

,0
0,True
1,True
2,True
3,False
4,False
5,True


In [20]:
s[s.notnull()] # chama o método notnull() diretamente na própria Series.

,0
0,1.0
1,2.0
2,3.0
5,4.0


---

##**Dropping null values**

Boolean selection + notnull() seems a little bit verbose and repetitive. Any repetitive task will probably have a better, more DRY way. In this case, we can use the `dropna` method:

In [21]:
s.dropna() # excluir valores que estão faltando

,0
0,1.0
1,2.0
2,3.0
5,4.0


Não estamos alterando a Series original

##**Dropping null values on DataFrames**
With DataFrames you can't drop single values. You cna only drop entire columns or rows.

In [22]:
df = pd.DataFrame({
    'Column A': [1, np.nan, 30, np.nan],
    'Column B': [2, 8, 31, np.nan],
    'Column C': [np.nan, 9, 32, 100],
    'Column D': [5, 8, 34, 110],
})

In [23]:
df

,Column A,Column B,Column C,Column D
0,1.0,2.0,NaN,5
1,NaN,8.0,9.0,8
2,30.0,31.0,32.0,34
3,NaN,NaN,100.0,110


In [24]:
df.shape

(4, 4)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Column A  2 non-null      float64
 1   Column B  3 non-null      float64
 2   Column C  3 non-null      float64
 3   Column D  4 non-null      int64  
dtypes: float64(3), int64(1)
memory usage: 260.0 bytes


In [26]:
df.isnull( )

,Column A,Column B,Column C,Column D
0,False,False,True,False
1,True,False,False,False
2,False,False,False,False
3,True,True,False,False


In [27]:
df.isnull().sum() # mostra quantos valores nulos há em cada coluna

,0
Column A,2
Column B,1
Column C,1
Column D,0


In [28]:
df.dropna()

,Column A,Column B,Column C,Column D
2,30.0,31.0,32.0,34


In this case we're dropping **rows**. Rows containing null values are dropped from the DF. You can aslo use the `axis` parameter to drop columns containing null values.

In [29]:
df.dropna(axis=1)  #axis='columns' also works

,Column D
0,5
1,8
2,34
3,110


Any row or column will be dropped. Which can be, depending on the case, too extreme. You can control this behavior with the `how` parameter. Can be either `'any'` or `'all'`:

In [30]:
df2 = pd.DataFrame({
    'Column A': [1, np.nan, 30],
    'Column B': [2, np.nan, 31],
    'Column C': [np.nan, np.nan, 100]
})

In [31]:
df2

,Column A,Column B,Column C
0,1.0,2.0,NaN
1,NaN,NaN,NaN
2,30.0,31.0,100.0


In [32]:
df2.dropna(how='all') # remove a linha com todos os valores nulos

,Column A,Column B,Column C
0,1.0,2.0,NaN
2,30.0,31.0,100.0


In [33]:
df.dropna(how='any') # default behavior

,Column A,Column B,Column C,Column D
2,30.0,31.0,32.0,34


You can also use the thresh parameter to indicate a threshold(a minimum number) of non-null values for the row/column to be kept:

In [34]:
df

,Column A,Column B,Column C,Column D
0,1.0,2.0,NaN,5
1,NaN,8.0,9.0,8
2,30.0,31.0,32.0,34
3,NaN,NaN,100.0,110


In [35]:
df.dropna(thresh=3)

,Column A,Column B,Column C,Column D
0,1.0,2.0,NaN,5
1,NaN,8.0,9.0,8
2,30.0,31.0,32.0,34


In [36]:
df.dropna(thresh=3, axis='columns')

,Column B,Column C,Column D
0,2.0,NaN,5
1,8.0,9.0,8
2,31.0,32.0,34
3,NaN,100.0,110


---

##**Filling null values**

Instead than dropping the null values, we can might need to replace them with some other value. Sometimes a `nan` can be replaced with a `0`, sometimes it can be replaced with the `mean` of the sample, and some other times you can take the closest value. Again, it depends on the context.

In [37]:
s

,0
0,1.0
1,2.0
2,3.0
3,NaN
4,NaN
5,4.0


**Filling nulls with a arbitrary value**

In [38]:
s.fillna(0)

,0
0,1.0
1,2.0
2,3.0
3,0.0
4,0.0
5,4.0


In [39]:
s.fillna(s.mean())

,0
0,1.0
1,2.0
2,3.0
3,2.5
4,2.5
5,4.0


In [40]:
s # não está alterando a série original

,0
0,1.0
1,2.0
2,3.0
3,NaN
4,NaN
5,4.0


**Filling nulls with contiguos (close) values**

The method argument is used to fill null values with other values close to that null one:

In [41]:
s.fillna(method='ffill') # forward fill

/tmp/ipykernel_350/2500342873.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  s.fillna(method='ffill') # forward fill


,0
0,1.0
1,2.0
2,3.0
3,3.0
4,3.0
5,4.0


In [42]:
s.fillna(method='bfill') #backwards fill

/tmp/ipykernel_350/3469294209.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  s.fillna(method='bfill') #backwards fill


,0
0,1.0
1,2.0
2,3.0
3,4.0
4,4.0
5,4.0


This can still leave null values at the extremes of the Series/DataFrame:

In [43]:
pd.Series([np.nan, 3, np.nan, 9]).fillna(method='ffill')

/tmp/ipykernel_350/533636215.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  pd.Series([np.nan, 3, np.nan, 9]).fillna(method='ffill')


,0
0,NaN
1,3.0
2,3.0
3,9.0


In [44]:
pd.Series([1, np.nan, 3, np.nan, np.nan]).fillna(method='bfill')

/tmp/ipykernel_350/2629518188.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  pd.Series([1, np.nan, 3, np.nan, np.nan]).fillna(method='bfill')


,0
0,1.0
1,3.0
2,3.0
3,NaN
4,NaN


**Filling null values on DataFrames**

The fillna method also works on DataFrames, and it works similarly

In [45]:
df

,Column A,Column B,Column C,Column D
0,1.0,2.0,NaN,5
1,NaN,8.0,9.0,8
2,30.0,31.0,32.0,34
3,NaN,NaN,100.0,110


In [46]:
df.fillna({'Column A': 0, 'Column B': 99, 'Column C': df['Column C'].mean()})

,Column A,Column B,Column C,Column D
0,1.0,2.0,47.0,5
1,0.0,8.0,9.0,8
2,30.0,31.0,32.0,34
3,0.0,99.0,100.0,110


In [47]:
df.fillna(method='ffill', axis=0) #vertical

/tmp/ipykernel_350/646670546.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', axis=0) #vertical


,Column A,Column B,Column C,Column D
0,1.0,2.0,NaN,5
1,1.0,8.0,9.0,8
2,30.0,31.0,32.0,34
3,30.0,31.0,100.0,110


In [48]:
df.fillna(method='ffill', axis=1) #horizontal

/tmp/ipykernel_350/3321250618.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', axis=1) #horizontal


,Column A,Column B,Column C,Column D
0,1.0,2.0,2.0,5.0
1,NaN,8.0,9.0,8.0
2,30.0,31.0,32.0,34.0
3,NaN,NaN,100.0,110.0


---

##**Checking of there are NAs**

The question is: Does this Series or DataFrame contain any missing values? The answer whould be yes or no: `True` or `False.` How can you verify it?

**Example 1: Checking the length**
If there are missing values, `s.dropna()` will have less elements than s:

In [49]:
s.dropna().count()

np.int64(4)

In [50]:
missing_values = len(s.dropna()) != len(s)
missing_values

True

There's also a `count` method, that excludes `nans` from its result:

In [51]:
len(s)

6

In [52]:
s.count()

np.int64(4)

So we could just do:

In [53]:
missing_values = s.count() != len(s)
missing_values

np.True_

**More Pythonic solution `any`**

The method `any` and `all` check if there's any True value in a Series or `all` values are `True`. They work in the same way as in Python:

In [54]:
pd.Series([True, False, False]).any()

np.True_

In [55]:
pd.Series([True, False, False]).all()

np.False_

In [56]:
pd.Series([True, True, True]).all()

np.True_

The isnull() method returned Boolen `Series` with `True` values wherever there was a `nan`:

In [57]:
s.isnull()

,0
0,False
1,False
2,False
3,True
4,True
5,False


So we can ajust the any method with the boolean array returned

In [58]:
pd.Series([1, np.nan]).isnull().any()

np.True_

In [60]:
pd.Series([1, 2]).isnull().any()

np.False_

In [61]:
s.isnull().any()

np.True_

A more strict version would check only the value of the Series:

In [62]:
s.isnull().values

array([False, False, False,  True,  True, False])

In [63]:
s.isnull().values.any()

np.True_

---